In [131]:
import pandas as pd
import numpy as np
import joblib
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, QED, rdFingerprintGenerator
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

In [ ]:
df = pd.read_csv('Datasets/tox21.csv')

df.head()

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [116]:
# df.info()

In [117]:
# df[df["NR-AR"] == 1.0].shape, df[df["NR-AR-LBD"] == 1.0].shape, df[df["NR-Aromatase"] == 1.0].shape, df[df["NR-ER"] == 1.0].shape

In [118]:
# df[df["NR-ER-LBD"] == 1.0].shape, df[df["NR-PPAR-gamma"] == 1.0].shape, df[df["SR-ARE"] == 1.0].shape, df[df["SR-ATAD5"] == 1.0].shape

In [119]:
# df[df["SR-HSE"] == 1.0].shape, df[df["SR-MMP"] == 1.0].shape, df[df["SR-p53"] == 1.0].shape

In [120]:
toxicity_columns = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 
    'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

df.dropna(subset=toxicity_columns, inplace=True)

In [121]:
df_clean = df.dropna(subset=['smiles']).copy()


In [122]:
df_clean.reset_index(inplace=True, drop=True)

# df_clean.head(10)

In [123]:

print(f"📊 Dataset: {len(df_clean)} compounds")

📊 Dataset: 3079 compounds


In [124]:
X_list = []
y_list = []

print(f"🔄 Starting feature extraction for {len(df_clean)} compounds...")

🔄 Starting feature extraction for 3079 compounds...


In [125]:
def extract_features_advanced(smiles, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    
    # 1. Fingerprint
    fp = morgan_gen.GetFingerprint(mol)
    fp_array = np.zeros((1,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, fp_array)
    
    # 2. Basic Properties (LogP, MW)
    logp = Descriptors.MolLogP(mol)
    mw = Descriptors.MolWt(mol)
    
    # 3. ADVANCED FEATURE from ZINC: QED (Drug-likeness)
    qed_score = QED.qed(mol) 
    
    # Now you have 2048 + 3 = 2051 features
    return np.append(fp_array, [logp, mw, qed_score])



In [128]:
X_list = []
y_list = []

print("🔄 Extracting features... skipping invalid molecules.")

# We iterate through the dataframe
for i, row in df_clean.iterrows():
    # Process the SMILES
    feat = extract_features_advanced(row['smiles'])
    
    if feat is not None:
        X_list.append(feat)
        # Capture ALL 12 labels for this molecule as a list
        # Note: We keep NaNs here; we will handle them during training
        y_list.append(row[toxicity_columns].values)

# Convert to final NumPy arrays
X = np.array(X_list)
Y = np.array(y_list).astype(float) # Shape: (Molecules, 12)

print(f"✅ Multi-Signal Extraction Complete.")
print(f"Features (X) shape: {X.shape}")
print(f"Labels (Y) shape: {Y.shape}")

🔄 Extracting features... skipping invalid molecules.


[10:38:29] Explicit valence for atom # 4 Al, 6, is greater than permitted
[10:38:30] Explicit valence for atom # 4 Al, 6, is greater than permitted
[10:38:31] Explicit valence for atom # 9 Al, 6, is greater than permitted
[10:38:31] Explicit valence for atom # 5 Al, 6, is greater than permitted
[10:38:31] Explicit valence for atom # 16 Al, 6, is greater than permitted


✅ Multi-Signal Extraction Complete.
Features (X) shape: (3074, 2051)
Labels (Y) shape: (3074, 12)


In [130]:
models_dict = {}
final_scores = {}

for idx, pathway_name in enumerate(toxicity_columns):
    y_pathway = Y[:, idx]
    valid_mask = ~np.isnan(y_pathway) 
    
    X_pathway = X[valid_mask]
    y_pathway_clean = y_pathway[valid_mask].astype(int)
    
    if sum(y_pathway_clean) < 10: # Skip if there are almost no toxic samples
        print(f"Skipping {pathway_name}: Insufficient toxic data.")
        continue

    # --- THE CRITICAL FIX: TRAIN/TEST SPLIT ---
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_pathway, y_pathway_clean, test_size=0.2, random_state=42, stratify=y_pathway_clean
    )

    pos_weight = (len(y_train_p) - sum(y_train_p)) / sum(y_train_p)
    
    model = XGBClassifier(
        n_estimators=150, # Lower estimators to prevent overfitting
        scale_pos_weight=pos_weight, 
        learning_rate=0.05,
        max_depth=3,      # Shallower trees are better for small datasets
        random_state=42
    )
    
    model.fit(X_train_p, y_train_p)
    
    # --- EVALUATE ON TEST DATA (The 'Real' Score) ---
    probs = model.predict_proba(X_test_p)[:, 1]
    score = roc_auc_score(y_test_p, probs)
    
    models_dict[pathway_name] = model
    final_scores[pathway_name] = score
    print(f"✅ {pathway_name} trained. Real Test AUC: {score:.4f} (Samples: {len(y_test_p)})")


✅ NR-AR trained. Real Test AUC: 0.7385 (Samples: 615)
✅ NR-AR-LBD trained. Real Test AUC: 0.5391 (Samples: 615)
✅ NR-AhR trained. Real Test AUC: 0.8203 (Samples: 615)
✅ NR-Aromatase trained. Real Test AUC: 0.7556 (Samples: 615)
✅ NR-ER trained. Real Test AUC: 0.6017 (Samples: 615)
✅ NR-ER-LBD trained. Real Test AUC: 0.6310 (Samples: 615)
✅ NR-PPAR-gamma trained. Real Test AUC: 0.8620 (Samples: 615)
✅ SR-ARE trained. Real Test AUC: 0.7104 (Samples: 615)
✅ SR-ATAD5 trained. Real Test AUC: 0.8230 (Samples: 615)
✅ SR-HSE trained. Real Test AUC: 0.5278 (Samples: 615)
✅ SR-MMP trained. Real Test AUC: 0.9003 (Samples: 615)
✅ SR-p53 trained. Real Test AUC: 0.7001 (Samples: 615)


In [ ]:
# Save the 12-model dictionary
joblib.dump(models_dict, 'Model/multi_pathway_toxicity_system.pkl')

['multi_pathway_toxicity_system.pkl']

In [ ]:
df_sample = pd.read_csv("Datasets/250k_rndm_zinc_drugs_clean.csv")
df_sample.head(2)

,smiles,logP,qed,SAS
0,CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1\n,5.0506,0.702012,2.084095
1,C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1\n,3.1137,0.928975,3.432004


In [134]:
zinc_logp_avg = df_sample['logP'].mean()
zinc_qed_avg = df_sample['qed'].mean()

print(f"ZINC Average LogP: {zinc_logp_avg}")
print(f"ZINC Average QED: {zinc_qed_avg}")

ZINC Average LogP: 2.4570930029063356
ZINC Average QED: 0.7282644882810482
